# Reshaping wide format data to long format

Going between wide and long format data is an important data manipulation skill. Different applications, like some of the visualization Python packages we'll explore later in the course, prefer data in long format. We've mostly been working with data in wide format.

* Wide data format: one variable for every column. Each observation has one row.
* Long data format: one column specifies the variable name, another specifies the value of that variable for an observation. Thus one observation can have multiple rows, one for each variable.

You guessed it. Long-form data has more rows and fewer columns than wide-form data!

Here's an example:

<img src="attachment:e333f86b-b99b-4643-8c4b-c12ea63772a4.png" width=500/>

*Image from [Statology](https://www.statology.org/long-vs-wide-data/)*

## Reshaping to long format using `pd.melt()`
We'll use the Pandas [`melt()`](https://pandas.pydata.org/docs/user_guide/reshaping.html#melt-and-wide-to-long) function to reshape data from wide to long format. The first thing to figure out when using `melt` is if there are any columns that act to identify the observation or "thing" that is being measured. These should not be gathered and stacked like other variables and are supplied to the `id_vars` argument of `melt`.

Let's try it out on a dataset about penguins.

<img src="attachment:c9c95fad-ff07-4980-9879-b078ea104654.png" width=500/>

*Image from [Pandas documentation](https://pandas.pydata.org/docs/user_guide/reshaping.html)*

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
penguins = sns.load_dataset('penguins')
penguins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [3]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


It looks like each row is a unique penguin! We will lose the index when we melt, so we sometimes want to keep the original index. Here we'll rename the index and then convert it to a regular column.

In [4]:
penguins.index.name = "penguin_id"
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
penguin_id,,,,,,,
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male


In [5]:
penguins.reset_index(inplace=True)
penguins

,penguin_id,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...,...
339,339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


Which columns should be the ID columns? Certainly It's sometimes hard to tell what should be, but which species and island the penguin are good choice for which "identify" the penguin than the numeric observations about the penguin. In practice, the identifier columns are often just the columns with categorical information (string values) instead of numeric, since we'll be plotting the numeric values separately.

In [6]:
penguins_long = penguins.melt(id_vars=["penguin_id", "species", "island", "sex"])
penguins_long

,penguin_id,species,island,sex,variable,value
0,0,Adelie,Torgersen,Male,bill_length_mm,39.1
1,1,Adelie,Torgersen,Female,bill_length_mm,39.5
2,2,Adelie,Torgersen,Female,bill_length_mm,40.3
3,3,Adelie,Torgersen,NaN,bill_length_mm,NaN
4,4,Adelie,Torgersen,Female,bill_length_mm,36.7
...,...,...,...,...,...,...
1371,339,Gentoo,Biscoe,NaN,body_mass_g,NaN
1372,340,Gentoo,Biscoe,Female,body_mass_g,4850.0
1373,341,Gentoo,Biscoe,Male,body_mass_g,5750.0
1374,342,Gentoo,Biscoe,Female,body_mass_g,5200.0


In [13]:
penguins_long.sample()

,penguin_id,species,island,sex,variable,value
123,123,Adelie,Torgersen,Male,bill_length_mm,41.4


Ok, we certainly have more columns now! We now have multiple rows for the same penguin. Each row is a numeric observation about the column with the name of the observation type, the **variable** column, and the **value** column for the value of that variable. Check the [Pandas guide to `melt()`](https://pandas.pydata.org/docs/user_guide/reshaping.html#melt-and-wide-to-long) to see how to change those column names if you wish.

How many rows do we have for each penguin?

In [14]:
penguins_long.penguin_id.value_counts().unique()

array([4], dtype=int64)

We still have 344 penguins, each is just repeated 4 times, one for each of the 4 numeric variables.